In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Predict 12-Month ED or Inpatient Visit Risk
# MAGIC
# MAGIC This notebook trains a binary classification model to predict whether a patient will have an ED visit or inpatient visit in the next 12 months.
# MAGIC
# MAGIC Target:
# MAGIC - `Outcome = 1`: ED or inpatient visit in next 12 months
# MAGIC - `Outcome = 0`: no ED or inpatient visit in next 12 months

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    DecimalType,
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
)
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Load Patient-Level Data

# COMMAND ----------

table_name = "hackathon.data.pophealth_pdf_train_patientlevel"

df = spark.table(table_name)

row_count = df.count()
col_count = len(df.columns)

print(f"Rows: {row_count:,}")
print(f"Columns: {col_count:,}")

display(df.limit(5))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Check Outcome Balance

# COMMAND ----------

outcome_counts = (
    df.groupBy("Outcome")
      .count()
      .withColumn("pct", F.round(F.col("count") / F.lit(row_count) * 100, 2))
      .orderBy("Outcome")
)

display(outcome_counts)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Prepare Features
# MAGIC
# MAGIC We exclude:
# MAGIC - `Outcome`, because it is the target
# MAGIC - `patientID`, because it is an identifier
# MAGIC - raw `EncMonth`, because we split it into year and month features

# COMMAND ----------

target_col = "Outcome"
id_cols = ["patientID"]
time_cols = ["EncMonth"]

df_model = (
    df
    .withColumn("EncYear", F.substring(F.col("EncMonth").cast("string"), 1, 4).cast("double"))
    .withColumn("EncMonthNum", F.substring(F.col("EncMonth").cast("string"), 5, 2).cast("double"))
    .withColumn(target_col, F.col(target_col).cast("double"))
)

exclude_cols = set([target_col] + id_cols + time_cols)
numeric_types = (IntegerType, LongType, DoubleType, FloatType, DecimalType)

numeric_cols = []
categorical_cols = []

for field in df_model.schema.fields:
    if field.name in exclude_cols:
        continue

Rows: 44,250
Columns: 417


DerivedRaceEthnicity,MaritalStatus,sex,CAHPIPercentile,CAHPIQuartile,ADINatrank,ADINatRankQuintile,ADIStaterank,straight_line_distance_miles,last_12months_officevisit_count,last_12months_officevisit_distinct_specialty,last_6months_officevisit_count,last_6months_officevisit_distinct_specialty,Specialty_internal_medicine,Specialty_family_practice,Specialty_im_fp_ob,Specialty_fam_prac_int_med,Specialty_fp_ob,Specialty_nephrology,Specialty_pulmonary,Specialty_cardiology,Specialty_oncology,Specialty_colorectal_oncology,Specialty_radiation_oncology,Specialty_obgyn_oncology,Specialty_surgical_oncology,last_12months_diagnosis_count_ccw,last_12months_chronic_count,cms_ccw_sensory_blindness_and_visual_impairment_36,cms_ccw_rheumatoid_arthritis_osteoarthritis_36,cms_ccw_hepatitis_b_acute_or_unspecified_36,cms_ccw_lung_cancer_36,cms_ccw_obesity_36,cms_ccw_colorectal_cancer_36,cms_ccw_ischemic_heart_disease_36,cms_ccw_post_traumatic_stress_disorder_ptsd_36,cms_ccw_female_male_breast_cancer_36,cms_ccw_hiv_aids_36,cms_ccw_endometrial_cancer_36,cms_ccw_pressure_and_chronic_ulcers_36,cms_ccw_stroke_transient_ischemic_attack_exclusion_36,cms_ccw_liver_disease_cirrhosis_and_other_liver_conditions_36,cms_ccw_acquired_hypothyroidism_36,cms_ccw_bipolar_disorder_36,cms_ccw_diabetes_36,cms_ccw_other_developmental_delays_36,cms_ccw_muscular_dystrophy_36,cms_ccw_hyperlipidemia_36,cms_ccw_depressive_disorders_36,cms_ccw_osteoporosis_36,cms_ccw_alcohol_use_disorders_36,cms_ccw_stroke_transient_ischemic_attack_36,cms_ccw_asthma_36,cms_ccw_heart_failure_36,cms_ccw_prostate_cancer_36,cms_ccw_cerebral_palsy_36,cms_ccw_sickle_cell_disease_36,cms_ccw_hepatitis_b_chronic_36,cms_ccw_migraine_and_chronic_headache_36,cms_ccw_cataract_36,cms_ccw_anxiety_disorders_36,cms_ccw_spina_bifida_and_other_congenital_anomalies_of_the_nervous_system_36,cms_ccw_multiple_sclerosis_and_transverse_myelitis_36,cms_ccw_adhd_conduct_disorders_and_hyperkinetic_syndrome_36,cms_ccw_hepatitis_d_36,cms_ccw_chronic_obstructive_pulmonary_disease_and_bronchietasis_36,cms_ccw_cystic_fibrosis_and_other_metabolic_developmental_disorders_36,cms_ccw_atrial_fibrillation_36,cms_ccw_hypertension_36,cms_ccw_traumatic_brain_injury_and_nonpsychotic_mental_disorders_due_to_brain_damage_36,cms_ccw_mobility_impairments_36,cms_ccw_epilepsy_36,cms_ccw_fibromyalgia_chronic_pain_and_fatigue_36,cms_ccw_glaucoma_36,cms_ccw_personality_disorders_36,cms_ccw_chronic_kidney_disease_36,cms_ccw_viral_hepatitis_general_36,cms_ccw_drug_use_disorders_36,cms_ccw_tobacco_use_36,cms_ccw_anemia_36,cms_ccw_depression_36,cms_ccw_peripheral_vascular_disease_pvd_36,cms_ccw_alzheimers_disease_36,cms_ccw_alzheimers_disease_and_related_disorders_or_senile_dementia_36,cms_ccw_spinal_cord_injury_36,cms_ccw_sensory_deafness_and_hearing_impairment_36,cms_ccw_schizophrenia_36,cms_ccw_autism_spectrum_disorders_36,cms_ccw_acute_myocardial_infarction_36,cms_ccw_benign_prostatic_hyperplasia_36,cms_ccw_hepatitis_c_chronic_36,cms_ccw_learning_disabilities_36,cms_ccw_leukemias_and_lymphomas_36,cms_ccw_schizophrenia_and_other_psychotic_disorders_36,cms_ccw_intellectual_disabilities_and_related_conditions_diagnoses_36,cms_ccw_hip_pelvic_fracture_36,cms_ccw_sensory_blindness_and_visual_impairment_60,cms_ccw_rheumatoid_arthritis_osteoarthritis_60,cms_ccw_hepatitis_b_acute_or_unspecified_60,cms_ccw_lung_cancer_60,cms_ccw_obesity_60,cms_ccw_colorectal_cancer_60,cms_ccw_ischemic_heart_disease_60,cms_ccw_post_traumatic_stress_disorder_ptsd_60,cms_ccw_female_male_breast_cancer_60,cms_ccw_hiv_aids_60,cms_ccw_endometrial_cancer_60,cms_ccw_pressure_and_chronic_ulcers_60,cms_ccw_stroke_transient_ischemic_attack_exclusion_60,cms_ccw_liver_disease_cirrhosis_and_other_liver_conditions_60,cms_ccw_acquired_hypothyroidism_60,cms_ccw_bipolar_disorder_60,cms_ccw_diabetes_60,cms_ccw_other_developmental_delays_60,cms_ccw_muscular_dystrophy_60,cms_ccw_hyperlipidemia_60,cms_ccw_depressive_disorders_60,cms_ccw_osteoporosis_60,cms_ccw_alcohol_use_disorders_60,cms_ccw_

Outcome,count,pct
0,37014,83.65
1,7236,16.35


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Standalone Model Outputs Export
# MAGIC
# MAGIC This notebook works even when run separately from the training notebook.
# MAGIC
# MAGIC It:
# MAGIC - loads the patient-level table
# MAGIC - recreates the same feature preparation
# MAGIC - loads the saved GBT pipeline model
# MAGIC - trains a quick logistic regression baseline
# MAGIC - exports AUROC/AUPRC, top 10% risk capture, top features, and risk scores

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    DecimalType,
)

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Load Data and Saved Model

# COMMAND ----------

table_name = "hackathon.data.pophealth_pdf_train_patientlevel"
model_path = "dbfs:/tmp/ed_inpatient_12mo_gbt_pipeline_model"

df = spark.table(table_name)
gbt_model = PipelineModel.load(model_path)

row_count = df.count()
print(f"Rows: {row_count:,}")
print(f"Columns: {len(df.columns):,}")
print(f"Loaded GBT model from: {model_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Prepare Features

# COMMAND ----------

target_col = "Outcome"
id_cols = ["patientID"]
time_cols = ["EncMonth"]

df_model = (
    df
    .withColumn("EncYear", F.substring(F.col("EncMonth").cast("string"), 1, 4).cast("double"))
    .withColumn("EncMonthNum", F.substring(F.col("EncMonth").cast("string"), 5, 2).cast("double"))
    .withColumn(target_col, F.col(target_col).cast("double"))
)

exclude_cols = set([target_col] + id_cols + time_cols)
numeric_types = (IntegerType, LongType, DoubleType, FloatType, DecimalType)

numeric_cols = []
categorical_cols = []

for field in df_model.schema.fields:
    if field.name in exclude_cols:
        continue
    if isinstance(field.dataType, numeric_types):
        numeric_cols.append(field.name)
    elif isinstance(field.dataType, StringType):
        categorical_cols.append(field.name)

for c in numeric_cols:
    df_model = df_model.withColumn(c, F.col(c).cast("double"))
    df_model = df_model.withColumn(
        f"{c}_is_missing",
        F.when(F.col(c).isNull(), 1.0).otherwise(0.0),
    )

print(f"Numeric columns: {len(numeric_cols):,}")
print(f"Categorical columns: {len(categorical_cols):,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Recreate Train/Test Split

# COMMAND ----------

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6054341800273454>, line 42
     39 model_path = "dbfs:/tmp/ed_inpatient_12mo_gbt_pipeline_model"
     41 df = spark.table(table_name)
---> 42 gbt_model = PipelineModel.load(model_path)
     44 row_count = df.count()
     45 print(f"Rows: {row_count:,}")

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/util.py:929, in MLReadable.load(cls, path)
    926 @classmethod
    927 def load(cls, path: str) -> RL:
    928     """Reads an ML instance from the input path, a shortcut of `read().load(path)`."""
--> 929     return cls.read().load(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/ml/connect/readwrite.py:228, in RemoteMLReader.load(self, path)
    225 session = SparkSession.getActiveSession()
    226 assert session is not None
--> 228 return RemoteMLReader.loadInstance(self._clazz, path, se